# Figure 4 DINOv3 visual clusters

This notebook retains the UMAP scatter plot, the park-level composition chart and the category-coloured word clouds. Run the section for the panel you need after the shared setup cell.

Input files are referenced by name; place the existing inputs in `INPUT_DIR` or point it to their location. Only figure files are saved if the plotting cells are run.


In [ ]:
from pathlib import Path
import pandas as pd

# Point this to the existing input files for the selected panel.
INPUT_DIR = Path(".")
OUTPUT_DIR = Path("figure_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Panel a UMAP scatter plot

Source: `DINOv3_Embedding_Batch_Merge_and_Clustering.ipynb`, cell 35, section 8.4. The plotting block matches the saved K6 UMAP graphic. Point `UMAP_INPUT` to the final filtered coordinates from the study. This cell does not recompute embeddings, PCA, clustering, or UMAP.


In [ ]:
import matplotlib.pyplot as plt
UMAP_INPUT = INPUT_DIR / "all_image_level_joint_clusters_k6.csv"
label_col = "kmeans_k6"
K_FINAL = 6
OUT_DIR = OUTPUT_DIR
all_df = pd.read_csv(UMAP_INPUT, usecols=["umap_x", "umap_y", label_col])
if len(all_df) != 328342:
    raise ValueError("Use the 328,342-image manuscript input, not the earlier unfiltered export.")


In [ ]:
# ------------------------------------------------------------
# 8.4 Plot the UMAP clusters.
# ------------------------------------------------------------
fig_dir_k = OUT_DIR / f"figures_k{K_FINAL}"
fig_dir_k.mkdir(exist_ok=True)

plt.figure(figsize=(10, 8))
plt.scatter(
    all_df["umap_x"],
    all_df["umap_y"],
    c=all_df[label_col],
    s=2,
    alpha=0.6
)
plt.title(f"Joint DINOv3 Image Clustering: KMeans k={K_FINAL}")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.tight_layout()

umap_png = fig_dir_k / f"umap_kmeans_k{K_FINAL}.png"
plt.savefig(umap_png, dpi=300)
plt.show()

print("Saved UMAP figure:", umap_png)


## Panel b Park-level proportions

Source: `DINOv3_K6_Park_Composition_Visualisation.ipynb`, cell 3. Read the saved 31-park K6 proportion workbook.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
bar_data = pd.read_excel(INPUT_DIR / "park_cluster_proportions_k6_20260517_173924.xlsx", index_col="park_name")
bar_data.columns = bar_data.columns.map(str)
cluster_cols = [str(i) for i in range(6)]
bar_data = bar_data[cluster_cols]
if len(bar_data) != 31:
    raise ValueError("Expected the 31-park proportion table.")
output_dir = OUTPUT_DIR
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")


In [ ]:
# =========================
# 8. Draw the stacked bar chart using the existing code and new data.
# =========================

fig, ax = plt.subplots(figsize=(14, 7))

bottom = np.zeros(len(bar_data))

# Use a colormap for the bars
colors = plt.cm.get_cmap('viridis', len(cluster_cols))

for i, c in enumerate(cluster_cols):
    ax.bar(
        bar_data.index,
        bar_data[c],
        bottom=bottom,
        label=c,
        color=colors(i) # Assign color from colormap
    )
    bottom += bar_data[c].values

ax.set_ylabel("Cluster proportion")
ax.set_xlabel("Parks")
ax.set_title("Park-level Visual Cluster Composition (K=6)")
ax.set_ylim(0, 1)

plt.xticks(rotation=90)
ax.legend(
    title="Cluster",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

bar_output = output_dir / f"park_cluster_ratio_stacked_bar_k6_{timestamp}.png"
plt.savefig(bar_output, dpi=300, bbox_inches="tight")
plt.show()

print("堆叠柱状图已保存:", bar_output)


## Panel c Cluster word clouds

Source: `Evaluate_DINOv3_Clusters_with_RAM_Tags.ipynb`, cells 34 and 35. Use the saved K6 tag-frequency export and original category mapping. The system font setup is optional; install the font beforehand if needed.


In [ ]:
import pandas as pd
from wordcloud import WordCloud
import numpy as np

# 1. Define the category to hex color mapping based on user's request
CATEGORY_COLORS = {
    'G1': '#6BA368', # Green.
    'G2': '#5B8DB8', # Blue.
    'G3': '#A0A4A8', # Grey.
    'G4': '#D8A47F', # Light brown / skin tone.
    'G5': '#C06C84', # Magenta.
    'G6': '#7A6C9D', # Purple.
    'G7': '#8AA6A3', # Grey-green.
    'G8': '#C9C9A3', # Pale yellow-green.
    'Uncategorized (未分类)': '#000000' # Black for uncategorized words
}

# 2. Load the word categorization data from Excel
excel_path = INPUT_DIR / "2625词根据词云更新.xlsx"

try:
    df_categories = pd.read_excel(excel_path)
    print(f"已成功加载词语分类文件: {excel_path}")
    # Use columns '词语' (word) and '分类' (category) for the word-to-category mapping.
    word_to_category_map = df_categories.set_index('词语')['分类'].to_dict()
    print(f"已从 Excel 文件中构建 {len(word_to_category_map)} 个词语的分类映射。")
except FileNotFoundError:
    print(f"错误: 词语分类文件 '{excel_path}' 未找到。词云将无法按分类着色，所有词语将显示为 'Uncategorized' 颜色。")
    word_to_category_map = {}
except KeyError as e:
    print(f"错误: 词语分类文件中缺少预期的列。请确保文件包含 '词语' 和 '分类' 列。错误: {e}")
    word_to_category_map = {}
except Exception as e:
    print(f"加载词语分类文件时发生错误: {e}")
    word_to_category_map = {}

# 3. Create a custom color function for WordCloud
# This function will be passed to the `color_func` parameter of WordCloud
# It tries to find the word's category and then its color. If not found, it defaults to 'Uncategorized'.
def custom_color_func(word, font_size, position, orientation, random_state=None, **kwargs):
    category = word_to_category_map.get(word, 'Uncategorized (未分类)')
    hex_color = CATEGORY_COLORS.get(category, CATEGORY_COLORS['Uncategorized (未分类)'])
    # WordCloud expects an RGB tuple, so convert hex to RGB
    hex_color = hex_color.lstrip('#')
    rgb_color = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return rgb_color # Corrected: Return RGB tuple instead of string

print("词云着色功能已准备就绪。")


In [ ]:
import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import os
import datetime
import matplotlib.font_manager

output_directory = str(OUTPUT_DIR)
ANALYSIS_TIMESTAMP_K6 = "20260517_182607"
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = '/usr/share/fonts/truetype/wqy/wqy-microhei.ttc'
chinese_font_name = 'WenQuanYi Micro Hei' # Default name for this font

if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)

    found_font = None
    # Try to find the exact registered name for the WQY font
    for font in fm.fontManager.ttflist:
        if os.path.basename(font.fname) == os.path.basename(font_path):
            found_font = font.name
            break

    if found_font:
        print(f"字体 '{font_path}' (注册名: '{found_font}') 已添加到 matplotlib 字体管理器。")
        # Set font properties for matplotlib to display Chinese characters
        plt.rcParams['font.sans-serif'] = [found_font, 'SimHei'] + plt.rcParams['font.sans-serif'] # Prepend
        plt.rcParams['font.family'] = 'sans-serif' # Explicitly set font family
        print(f"matplotlib rcParams['font.sans-serif'] 已设置为: {plt.rcParams['font.sans-serif']}")
    else:
        print(f"警告: 无法在 fontManager 中找到匹配的中文系统字体 '{font_path}'。将尝试使用 'SimHei' 作为备用。")
        plt.rcParams['font.sans-serif'] = ['SimHei'] + plt.rcParams['font.sans-serif']
        plt.rcParams['font.family'] = 'sans-serif'
        print(f"使用备用字体 SimHei 设置 matplotlib rcParams['font.sans-serif']。")

else:
    print(f"警告: 中文字体文件 '{font_path}' 不存在。词云可能无法正确显示中文字符。")
    font_path = None

plt.rcParams['axes.unicode_minus'] = False # Correctly display minus signs

input_csv_filename = f'all_tags_per_cluster_kmeans_k6_{ANALYSIS_TIMESTAMP_K6}.csv'
input_csv_path = INPUT_DIR / input_csv_filename

try:
    df_all_tags = pd.read_csv(input_csv_path)
    print(f"已成功从 '{input_csv_path}' 加载所有标签频率数据。")
except FileNotFoundError:
    print(f"错误: 未找到文件 '{input_csv_path}'。请确保 `3add444f` 单元格已运行并生成了该文件。")
    exit()
except Exception as e:
    print(f"加载文件 '{input_csv_path}' 时发生错误: {e}")
    exit()

# Get unique cluster IDs
cluster_ids = sorted(df_all_tags['cluster'].unique())

for cluster_id in cluster_ids:
    print(f"正在为 kmeans_k6 聚类 {cluster_id} 生成词云...")

    # Filter data for the current cluster
    cluster_data = df_all_tags[df_all_tags['cluster'] == cluster_id]

    # Create a dictionary of word frequencies
    word_freq = dict(zip(cluster_data['tag'], cluster_data['count']))

    # --- DIAGNOSTIC START: Check categorized vs uncategorized words ---
    categorized_words_count = 0
    uncategorized_words_count = 0
    for word in word_freq.keys():
        if word in word_to_category_map:
            categorized_words_count += 1
        else:
            uncategorized_words_count += 1
    print(f"聚类 {cluster_id} 中有 {categorized_words_count} 个词语已分类，{uncategorized_words_count} 个词语未分类。")
    if (categorized_words_count + uncategorized_words_count) > 0 and uncategorized_words_count > 0:
        print("未分类词语占总词语的 {:.2f}%。".format(uncategorized_words_count / (categorized_words_count + uncategorized_words_count) * 100))
    # --- DIAGNOSTIC END ---

    # Generate word cloud (including uncategorized for full view)
    wc = WordCloud(
        background_color='white',
        width=800, height=400,
        max_words=50, # Limit the number of words in the cloud
        font_path=font_path, # Use the specified font for Chinese characters
        collocations=False, # Set to False to avoid duplicate words if tags are phrases
        color_func=custom_color_func # Apply the custom color function
    )
    wc.generate_from_frequencies(word_freq)

    # Plot and save the word cloud
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.title(f'kmeans_k6 Cluster {cluster_id} Word Cloud (All Words)', fontsize=15)
    plt.axis('off')

    output_image_filename = f'wordcloud_kmeans_k6_cluster_{cluster_id}_all_words_{ANALYSIS_TIMESTAMP_K6}.png'
    output_image_path = os.path.join(output_directory, output_image_filename)
    plt.savefig(output_image_path, bbox_inches='tight')
    print(f"词云图 (所有词语) 已保存到: {output_image_path}")
    plt.show()
    plt.close() # Close the figure to free memory

print("所有聚类的词云生成完成。")
